## Speculative Decoding

In this notebook, I will be testing to perform speculative decoding following the implementation by [Leviathan et al. (2022)](https://arxiv.org/pdf/2211.17192) in *Fast Inference from Transformers via Speculative Decoding*.


Speculative decoding addresses a key bottleneck in LLM inference: the sequential nature of autoregressive generation. Since each token requires a complete forward pass through the large model, and tokens must be generated one-by-one, this process is inherently slow. Speculative decoding speeds this up by using a smaller draft model to predict multiple tokens ahead, then verifying them in parallel with the large model.

In [34]:
! pip install --quiet huggingface_hub

In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [36]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

In order to demonstrate speculative decoding, we need a target model ($M_p$) and a draft model ($M_q$). I will be using Llama 3.2-8B Instruct as $M_p$ and Llama 3.2-1B Instruct as $M_q$.

In [37]:
target_model = "gpt2-medium"  # 355M parameters
draft_model = "gpt2"  # 124M parameters

tokenizer = AutoTokenizer.from_pretrained(target_model)
tokenizer.pad_token = tokenizer.eos_token

mp = AutoModelForCausalLM.from_pretrained(
    target_model,
    dtype=torch.float16,
    device_map="auto"
)

mq = AutoModelForCausalLM.from_pretrained(
    draft_model,
    dtype=torch.float16,
    device_map="auto"
)

The `speculative_decode` function performs speculative decoding given a prompt, the target and draft models, and $\gamma$.

First, we generate $\gamma$ number of tokens from the draft model, $M_q$. We record the probabilities of these tokens. In one pass, we then perform a verification step to determine the probabilities of these $\gamma$ tokens using the target model $

In [43]:
def sample_token(logits, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1), probs

def speculative_decode(prompt, mp, mq, tokenizer, gamma=4, max_tokens=50, temperature=1.0):
    # Get device from model (handle both single device and device_map cases)
    device = next(mp.parameters()).device

    # Tokenize initial prompt and move to correct device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated_tokens = 0
    total_draft_tokens = 0
    total_accepted_tokens = 0
    token_decisions = []  # Track detailed decisions for each token
    mp_calls = 0  # Count target model forward passes
    mq_calls = 0  # Count draft model forward passes

    with torch.no_grad():
        while generated_tokens < max_tokens:
            # Step 1: Use draft model to generate gamma tokens
            draft_tokens = []
            draft_probs = []
            current_input = input_ids.clone()

            for _ in range(gamma):
                outputs = mq(current_input)
                mq_calls += 1
                logits = outputs.logits[:, -1, :]  # Get logits for last position
                next_token, probs = sample_token(logits, temperature)

                draft_tokens.append(next_token)
                draft_probs.append(probs)

                # Append token for next iteration (keep on same device)
                # next_token is shape [batch_size], need to make it [batch_size, 1]
                current_input = torch.cat([current_input, next_token.unsqueeze(-1)], dim=-1)

            total_draft_tokens += gamma

            # Step 2: Verify all draft tokens in parallel with target model
            # Run target model on input + all draft tokens
            draft_tokens_tensor = torch.stack([t.squeeze() for t in draft_tokens]).unsqueeze(0)  # [1, gamma]
            verification_input = torch.cat([input_ids, draft_tokens_tensor], dim=-1)
            target_outputs = mp(verification_input)
            mp_calls += 1
            target_logits = target_outputs.logits[0, -gamma-1:-1, :]  # Get logits for draft positions

            # Step 3: Accept/reject tokens
            accepted_count = 0
            for i in range(gamma):
                draft_token = draft_tokens[i]
                target_probs = F.softmax(target_logits[i] / temperature, dim=-1)

                # Get probabilities for the drafted token
                p_target = target_probs[draft_token].item()
                p_draft = draft_probs[i][0, draft_token].item()

                # Acceptance probability: min(1, p_target / p_draft)
                accept_prob = min(1.0, p_target / p_draft)

                # Sample random value for acceptance decision
                random_val = torch.rand(1).item()
                accepted = random_val < accept_prob

                # Get token string for logging
                token_str = tokenizer.decode([draft_token.item()])

                # Record decision details
                decision = {
                    'iteration': len(token_decisions),
                    'token_id': draft_token.item(),
                    'token_str': token_str,
                    'q_prob': p_draft,
                    'p_prob': p_target,
                    'accept_prob': accept_prob,
                    'random_val': random_val,
                    'accepted': accepted,
                    'draft_position': i,
                }

                # Accept with probability accept_prob
                if accepted:
                    # Accept this token
                    input_ids = torch.cat([input_ids, draft_token.reshape(1, 1)], dim=-1)
                    accepted_count += 1
                    total_accepted_tokens += 1
                    generated_tokens += 1
                    token_decisions.append(decision)
                else:
                    # Reject: sample from adjusted distribution
                    # Adjusted distribution: max(0, p_target - p_draft) normalized
                    adjusted_probs = torch.clamp(target_probs - draft_probs[i], min=0.0)
                    adjusted_probs = adjusted_probs / adjusted_probs.sum()

                    new_token = torch.multinomial(adjusted_probs, num_samples=1)
                    new_token_str = tokenizer.decode([new_token.item()])

                    # Update decision with sampled token info
                    decision['sampled_from_adjusted'] = True
                    decision['adjusted_token_id'] = new_token.item()
                    decision['adjusted_token_str'] = new_token_str
                    token_decisions.append(decision)

                    input_ids = torch.cat([input_ids, new_token], dim=-1)
                    generated_tokens += 1
                    break  # Stop after first rejection

            # Step 4: If all tokens accepted, sample one bonus token
            if accepted_count == gamma:
                bonus_logits = target_outputs.logits[0, -1, :]
                bonus_token, _ = sample_token(bonus_logits, temperature)
                bonus_token_str = tokenizer.decode([bonus_token.item()])

                # Record bonus token
                token_decisions.append({
                    'iteration': len(token_decisions),
                    'token_id': bonus_token.item(),
                    'token_str': bonus_token_str,
                    'q_prob': None,  # No draft model prediction
                    'p_prob': None,  # Sampled directly from target
                    'accept_prob': None,
                    'random_val': None,
                    'accepted': True,
                    'draft_position': None,
                    'bonus_token': True,
                })

                # bonus_token might be scalar after squeeze, so reshape properly
                input_ids = torch.cat([input_ids, bonus_token.reshape(1, 1)], dim=-1)
                generated_tokens += 1

            # Check if we hit EOS token
            if input_ids[0, -1] == tokenizer.eos_token_id:
                break

    # Decode generated text
    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

    # Calculate statistics
    stats = {
        "total_generated_tokens": generated_tokens,
        "total_draft_tokens": total_draft_tokens,
        "total_accepted_tokens": total_accepted_tokens,
        "acceptance_rate": total_accepted_tokens / total_draft_tokens if total_draft_tokens > 0 else 0,
        "mp_calls": mp_calls,
        "mq_calls": mq_calls,
    }

    return generated_text, stats, token_decisions

In [62]:
prompt = "The future of artificial intelligence is"
result, stats, token_decisions = speculative_decode(
    prompt=prompt,
    mp=mp,
    mq=mq,
    tokenizer=tokenizer,  # Changed from target_tokenizer
    gamma=4,
    max_tokens=10,
    temperature=1.0
)

print("Generated text:")
print(result)
print("\nStatistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print("\n" + "="*80)
print("TOKEN DECISIONS:")
print("="*80)
for decision in token_decisions:
    if decision.get('bonus_token'):
        print(f"\n[BONUS TOKEN]")
        print(f"  Token: '{decision['token_str']}' (id={decision['token_id']})")
    else:
        print(f"\n[Token #{decision['iteration']}] Draft position: {decision['draft_position']}")
        print(f"  Token: '{decision['token_str']}' (id={decision['token_id']})")
        print(f"  q(x) = {decision['q_prob']:.6f}  (draft model probability)")
        print(f"  p(x) = {decision['p_prob']:.6f}  (target model probability)")
        print(f"  Accept prob = min(1, p/q) = {decision['accept_prob']:.6f}")
        print(f"  Random value = {decision['random_val']:.6f}")
        print(f"  Decision: {'✓ ACCEPTED' if decision['accepted'] else '✗ REJECTED'}")

        if not decision['accepted']:
            print(f"  → Sampled from adjusted distribution: '{decision['adjusted_token_str']}' (id={decision['adjusted_token_id']})")

Generated text:
The future of artificial intelligence is never clear, Singularity Hub owner Eduardo S

Statistics:
  total_generated_tokens: 10
  total_draft_tokens: 20
  total_accepted_tokens: 5
  acceptance_rate: 0.25
  mp_calls: 5
  mq_calls: 20

TOKEN DECISIONS:

[Token #0] Draft position: 0
  Token: ' never' (id=1239)
  q(x) = 0.002064  (draft model probability)
  p(x) = 0.002653  (target model probability)
  Accept prob = min(1, p/q) = 1.000000
  Random value = 0.302176
  Decision: ✓ ACCEPTED

[Token #1] Draft position: 1
  Token: ' going' (id=1016)
  q(x) = 0.078674  (draft model probability)
  p(x) = 0.044830  (target model probability)
  Accept prob = min(1, p/q) = 0.569822
  Random value = 0.652247
  Decision: ✗ REJECTED
  → Sampled from adjusted distribution: ' clear' (id=1598)

[Token #2] Draft position: 0
  Token: ',' (id=11)
  q(x) = 0.364258  (draft model probability)
  p(x) = 0.286377  (target model probability)
  Accept prob = min(1, p/q) = 0.786193
  Random value = 0.

### Time Comparision

In [54]:
def standard_autoregressive_decode(prompt, model, tokenizer, max_tokens=50, temperature=1.0):
    device = next(model.parameters()).device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated_tokens = 0
    model_calls = 0

    with torch.no_grad():
        while generated_tokens < max_tokens:
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :]
            next_token, _ = sample_token(logits, temperature)

            input_ids = torch.cat([input_ids, next_token.reshape(1, 1)], dim=-1)
            generated_tokens += 1
            model_calls += 1

            if input_ids[0, -1] == tokenizer.eos_token_id:
                break

    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

    stats = {
        "total_tokens": generated_tokens,
        "model_calls": model_calls,
    }

    return generated_text, stats

# Benchmark comparison function
def compare_methods(prompt, mp, mq, tokenizer, gamma=4, max_tokens=50, temperature=1.0, num_runs=3):
    print("="*80)
    print("BENCHMARK: Speculative Decoding vs Standard Autoregressive")
    print("="*80)
    print(f"Prompt: '{prompt}'")
    print(f"Max tokens: {max_tokens}, Gamma: {gamma}, Temperature: {temperature}")
    print(f"Averaging over {num_runs} runs...\n")

    # Warm up GPU
    print("Warming up GPU...")
    with torch.no_grad():
        for _ in range(3):
            _ = standard_autoregressive_decode(prompt, mp, tokenizer, max_tokens=5, temperature=temperature)
            _ = speculative_decode(prompt, mp, mq, tokenizer, gamma=gamma, max_tokens=5, temperature=temperature)
            torch.cuda.synchronize()

    # Run standard autoregressive
    print(f"\nRunning standard autoregressive decoding ({num_runs} runs)...")
    standard_times = []
    standard_tokens_per_sec = []

    for i in range(num_runs):
        torch.cuda.synchronize()
        start_time = time.time()
        text, stats = standard_autoregressive_decode(prompt, mp, tokenizer, max_tokens, temperature)
        torch.cuda.synchronize()
        elapsed = time.time() - start_time

        tps = stats['total_tokens'] / elapsed if elapsed > 0 else 0
        standard_times.append(elapsed)
        standard_tokens_per_sec.append(tps)
        print(f"  Run {i+1}: {elapsed:.3f}s, {tps:.2f} tokens/s")

    avg_standard_time = sum(standard_times) / len(standard_times)
    avg_standard_tps = sum(standard_tokens_per_sec) / len(standard_tokens_per_sec)

    # Run speculative decoding
    print(f"\nRunning speculative decoding ({num_runs} runs)...")
    spec_times = []
    spec_tokens_per_sec = []
    acceptance_rates = []

    for i in range(num_runs):
        torch.cuda.synchronize()
        start_time = time.time()
        text, stats, decisions = speculative_decode(prompt, mp, mq, tokenizer, gamma, max_tokens, temperature)
        torch.cuda.synchronize()
        elapsed = time.time() - start_time

        tps = stats['total_generated_tokens'] / elapsed if elapsed > 0 else 0
        spec_times.append(elapsed)
        spec_tokens_per_sec.append(tps)
        acceptance_rates.append(stats['acceptance_rate'])
        print(f"  Run {i+1}: {elapsed:.3f}s, {tps:.2f} tokens/s, acceptance={stats['acceptance_rate']:.2%}")

    avg_spec_time = sum(spec_times) / len(spec_times)
    avg_spec_tps = sum(spec_tokens_per_sec) / len(spec_tokens_per_sec)
    avg_acceptance = sum(acceptance_rates) / len(acceptance_rates)

    # Calculate speedup
    speedup = avg_standard_time / avg_spec_time if avg_spec_time > 0 else 0

    # Print results
    print("\n" + "="*80)
    print("RESULTS:")
    print("="*80)
    print(f"\nStandard Autoregressive:")
    print(f"  Average time: {avg_standard_time:.3f}s")
    print(f"  Average tokens/sec: {avg_standard_tps:.2f}")

    print(f"\nSpeculative Decoding:")
    print(f"  Average time: {avg_spec_time:.3f}s")
    print(f"  Average tokens/sec: {avg_spec_tps:.2f}")
    print(f"  Average acceptance rate: {avg_acceptance:.2%}")

    print(f"\n{'🚀 SPEEDUP: ' + str(round(speedup, 2)) + 'x'}")
    print(f"Time saved: {((avg_standard_time - avg_spec_time) / avg_standard_time * 100):.1f}%")
    print("="*80)

    return {
        'standard_time': avg_standard_time,
        'standard_tps': avg_standard_tps,
        'speculative_time': avg_spec_time,
        'speculative_tps': avg_spec_tps,
        'acceptance_rate': avg_acceptance,
        'speedup': speedup,
    }


In [56]:
prompt = "The future of artificial intelligence is"

comparison = compare_methods(
    prompt=prompt,
    mp=mp,
    mq=mq,
    tokenizer=tokenizer,
    gamma=7,
    max_tokens=50,
    temperature=1.0,
    num_runs=3
)

BENCHMARK: Speculative Decoding vs Standard Autoregressive
Prompt: 'The future of artificial intelligence is'
Max tokens: 50, Gamma: 7, Temperature: 1.0
Averaging over 3 runs...

Warming up GPU...

Running standard autoregressive decoding (3 runs)...
  Run 1: 1.236s, 40.46 tokens/s
  Run 2: 1.003s, 37.90 tokens/s
  Run 3: 0.940s, 53.18 tokens/s

Running speculative decoding (3 runs)...
  Run 1: 1.786s, 28.00 tokens/s, acceptance=19.73%
  Run 2: 1.190s, 42.02 tokens/s, acceptance=36.73%
  Run 3: 1.112s, 46.77 tokens/s, acceptance=42.86%

RESULTS:

Standard Autoregressive:
  Average time: 1.060s
  Average tokens/sec: 43.85

Speculative Decoding:
  Average time: 1.362s
  Average tokens/sec: 38.93
  Average acceptance rate: 33.11%

🚀 SPEEDUP: 0.78x
Time saved: -28.6%


Based on these results, it appears that the acceptance rate $\alpha$ is quite low which is the contributing to speculative decoding actually being slower than using the target model.